In [9]:
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
import hdbscan

temp_df = pd.read_csv("/Users/fangsiyu/Desktop/Kolding Hackathon 2026/cluster_kmean_k3.csv")
X = temp_df.drop(columns=['timestamp','mov_net_timestamp', 'cluster_label'])
X = X.fillna(0)

# 【核心加入】SNN 與密度分群必須先經過特徵標準化，避免大數字霸凌
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Optuna 自動尋優核心目標函數 ---
def objective(trial):
    k = trial.suggest_int('k', 10, 50)
    eps = trial.suggest_int('eps', 1, k - 1)
    min_samples = trial.suggest_int('min_samples', 3, 15)
    
    # 計算 SNN 共享近鄰距離矩陣
    nn = NearestNeighbors(n_neighbors=k).fit(X_scaled)
    graph = nn.kneighbors_graph(X_scaled, mode='connectivity')
    snn_distance = k - (graph @ graph.T).toarray()
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    labels = dbscan.fit_predict(snn_distance)
    
    if len(set(labels)) < 2 or (len(set(labels)) == 2 and -1 in labels): 
        return -1.0
        
    try:
        dbcv_score = hdbscan.validity.validity_index(X_scaled, labels)
        return dbcv_score
    except ValueError:
        return -1.0

# 執行 Optuna 搜尋最佳參數
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100) # 黑客松時間有限，可先跑 100~200 輪看效果，沒問題再開大

print("--- DBSCAN + SNN, with Optuna 最佳參數 ---")
print(f"Best paras:         {study.best_params}")
print(f"Highest DBCV:       {study.best_value:.4f}")

# --- 用搜尋到的黃金參數重跑模型，並塞回你原本的 temp_df ---
best_k = study.best_params['k']
best_eps = study.best_params['eps']
best_min = study.best_params['min_samples']

# 用最佳 K 重建 SNN 距離
nn_best = NearestNeighbors(n_neighbors=best_k).fit(X_scaled)
graph_best = nn_best.kneighbors_graph(X_scaled, mode='connectivity')
snn_distance_best = best_k - (graph_best @ graph_best.T).toarray()

# 跑最終的精準分群
dbscan_best = DBSCAN(eps=best_eps, min_samples=best_min, metric='precomputed')
temp_df['cluster_label'] = dbscan_best.fit_predict(snn_distance_best)

# --- 你原本的輸出與存檔段放 ---
temp_df[['timestamp', 'cluster_label']]
temp_df.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/cluster_h_dbscan_k3.csv', index=False, encoding='utf-8-sig')
print("【成功】已利用 Optuna SNN-DBSCAN 完成最佳分群並成功導出！")

[I 2026-05-25 14:21:11,387] A new study created in memory with name: no-name-86ae04b4-1b37-4e44-949a-8fbd738ab230
[I 2026-05-25 14:21:11,392] Trial 0 finished with value: -1.0 and parameters: {'k': 29, 'eps': 4, 'min_samples': 11}. Best is trial 0 with value: -1.0.
[I 2026-05-25 14:21:11,395] Trial 1 finished with value: -1.0 and parameters: {'k': 32, 'eps': 4, 'min_samples': 9}. Best is trial 0 with value: -1.0.
[I 2026-05-25 14:21:11,400] Trial 2 finished with value: -0.06444970096947258 and parameters: {'k': 36, 'eps': 2, 'min_samples': 4}. Best is trial 2 with value: -0.06444970096947258.
[I 2026-05-25 14:21:11,402] Trial 3 finished with value: -1.0 and parameters: {'k': 14, 'eps': 11, 'min_samples': 6}. Best is trial 2 with value: -0.06444970096947258.
[I 2026-05-25 14:21:11,405] Trial 4 finished with value: -1.0 and parameters: {'k': 36, 'eps': 19, 'min_samples': 8}. Best is trial 2 with value: -0.06444970096947258.
[I 2026-05-25 14:21:11,408] Trial 5 finished with value: -1.0 an

--- DBSCAN + SNN, with Optuna 最佳參數 ---
Best paras:         {'k': 20, 'eps': 6, 'min_samples': 3}
Highest DBCV:       0.3864
【成功】已利用 Optuna SNN-DBSCAN 完成最佳分群並成功導出！


In [11]:
print(temp_df['cluster_label'].value_counts())

cluster_label
 1    72
-1    13
 0     4
Name: count, dtype: int64
